# CLIP Student：从导入图片到三组蒸馏实验

这是你的 **Student 端专用 notebook**。已经内嵌同学的交接包，不必再次上传 ZIP，也不需要重新训练或推理 Teacher。

**怎么用：** 上传此文件到 Google Colab →「运行时 / 更改运行时类型」选择 GPU（T4 即可）→ 按编号逐块运行。

- 已有：6000 条训练样本、400 条校准拟合样本、400 条验证选择样本、1000 条测试样本及 Teacher 输出。
- 本次完成：下载图片 → 校验 → 冻结 CLIP 提取特征 → CE / 原始 Jev 蒸馏 / 校准 Jev 蒸馏，各 3 个种子 → 最终测试。
- 只训练 MLP 分类头；CLIP 编码器冻结。Teacher 是 T4/FP16，本 notebook 的 Student 使用 FP32，单独记录环境。
- 图片源压缩包约 4.4 GB，需要网络及足够的 Google Drive 空间（建议至少 8 GB 可用）。首次还会下载 CLIP 权重。
- **未在 Colab GPU 上实跑。** 已做 notebook 代码静态检查与交接数据检查；实际下载、GPU 训练与效果以运行结果为准。

不要直接「全部运行」：安装后先重启会话；最终测试也有单独开关。不要看完测试结果后再修改本轮参数。


## 1．安装 Student 所需工具

保留 Colab 自带的 CUDA PyTorch，不重装 Teacher 的大型环境。安装固定版本的 CLIP 工具和数据处理依赖。

运行完本格后选择「运行时 → 重启会话」，**然后从第 2 节继续**。以后断线重连如环境被重置，重新运行安装和重启。


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.57.1", "huggingface_hub==0.36.0", "peft==0.17.1",
    "numpy==2.2.6", "pillow==11.3.0", "pandas==2.2.3", "matplotlib==3.10.6"], check=True)
print("安装完成。请重启会话，然后从第 2 节继续。")


## 2．连接 Google Drive，设置保存位置

授权挂载 Drive。训练结果保存在自己的 Drive，Colab 断线后仍保留。

默认路径可以直接用。同一轮实验不要修改目录、种子、软件版本或训练参数；续跑时会核对配置。临时图片存在 Colab 本机，断线后可从 Drive 的图片压缩包重新解压。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, hashlib, subprocess, shutil, zipfile, base64, io, math
import importlib.metadata as metadata
import torch
from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version('2.6'), '此固定 CLIP 权重要求 PyTorch >= 2.6；请使用更新的 Colab GPU 运行时。'
from transformers import CLIPModel, CLIPProcessor  # 提前检查依赖，避免下载图片后才遇到导入失败
assert torch.cuda.is_available(), "请先在 Colab 中选择 GPU 运行时。"
WORK = Path('/content/drive/MyDrive/CLIP_Jev_Student_v1')
INPUT = WORK / 'handoff'                 # 原始交接材料，不在这里写训练结果
FEATURES = WORK / 'features'             # 可复用的 CLIP 特征
STUDENTS = WORK / 'students'             # 模型和断点
RESULTS = WORK / 'student_results'       # Student 独立报告
IMAGES = Path('/content/clip_jev_images') # 临时图片；不占重复的 Drive 空间
for folder in [WORK, INPUT, FEATURES, RESULTS, IMAGES]:
    folder.mkdir(parents=True, exist_ok=True)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def read_rows(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

def save_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.partial')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)

def run(*args):
    subprocess.run([sys.executable, '-u', *map(str, args)], cwd=INPUT, check=True)

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
versions = {n: metadata.version(n) for n in ['torch','transformers','huggingface_hub','peft','numpy','pillow']}
environment = dict(versions=versions, gpu=torch.cuda.get_device_name(0),
                   cuda=torch.version.cuda, student_dtype='float32', frozen_clip=True)
env_path = WORK / 'student_environment.json'
if env_path.exists():
    assert json.loads(env_path.read_text()) == environment, "运行环境变了；恢复相同环境，或使用新的 WORK 目录。"
else:
    save_json(env_path, environment)
print('保存目录：', WORK)
print('GPU：', environment['gpu'], '；Student 精度：FP32')


## 3．导入同学的实验包（已内嵌，直接运行）

本格包含原压缩包的数据，代码较长，无需编辑。先验证整个包的 SHA256，再逐个恢复文件；遇到已存在但内容不同的文件会停止，避免混合实验。


In [ ]:
PACKAGE_SHA256 = '0d0106e0833aaebc944cf1048876d162a04a4f83a5696539573928d66f3de3a4'
PACKAGE_B64 = '...'
package_bytes = base64.b64decode(PACKAGE_B64)
assert hashlib.sha256(package_bytes).hexdigest() == PACKAGE_SHA256
with zipfile.ZipFile(io.BytesIO(package_bytes)) as z:
    for item in z.infolist():
        if item.is_dir():
            continue
        dest = (INPUT / item.filename).resolve()
        assert dest.is_relative_to(INPUT.resolve()), 'ZIP 路径超出交接目录'
        content = z.read(item)
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists():
            assert dest.read_bytes() == content, f'已有交接文件发生变化：{item.filename}'
        else:
            temp = dest.with_suffix(dest.suffix + '.partial')
            temp.write_bytes(content)
            temp.replace(dest)
sys.path.insert(0, str(INPUT))
print('交接包已恢复并校验。', PACKAGE_SHA256)


## 4．检查训练资料是否对齐

检查样本数量、图片划分隔离、Teacher logits、标签、图片哈希、校准来源。**不重新抽样、不重新拟合校准参数。** 测试清单在这里仅用于完整性核对，不用于训练或选择模型。


In [ ]:
EXPECTED = {'train':6000, 'val_fit':400, 'val_select':400, 'test':1000}
manifest = {s:read_rows(INPUT/'manifests'/f'{s}.jsonl') for s in EXPECTED}
audit_rows = read_rows(INPUT/'manifests/image_audit.jsonl')
audit = {r['sample_id']:r for r in audit_rows}
assert len(audit_rows) == len(audit) == 7800
calibration = json.loads((INPUT/'results/calibration.json').read_text())
assert calibration['labels'] == ['E','N','C']
seen_ids, seen_images = set(), set()
for split, expected in EXPECTED.items():
    rows = manifest[split]
    ids = {r['sample_id'] for r in rows}
    images = {r['image_id'] for r in rows}
    assert len(rows) == len(ids) == len(images) == expected
    assert not (ids & seen_ids) and not (images & seen_images)
    seen_ids.update(ids); seen_images.update(images)
    assert all(r['label'] in [0,1,2] and r['sample_id'] in audit for r in rows)
    if split == 'test':
        continue
    record = json.loads((INPUT/'predictions'/split/'run.json').read_text())
    assert record['saved_order'] == ['E','N','C']
    assert record['manifest_sha256'] == sha(INPUT/'manifests'/f'{split}.jsonl')
    assert record['image_audit_sha256'] == sha(INPUT/'manifests/image_audit.jsonl')
    for model in ['base','jev']:
        path = INPUT/'predictions'/split/f'{model}.jsonl'
        predictions = read_rows(path)
        by_id = {r['sample_id']:r for r in predictions}
        assert len(predictions) == len(by_id) == expected and set(by_id) == ids
        for row in rows:
            pred = by_id[row['sample_id']]
            assert all(pred[k] == v for k,v in row.items())
            assert len(pred['logits']) == 3 and all(math.isfinite(v) for v in pred['logits'])
            assert pred['image_sha256'] == audit[row['sample_id']]['file_sha256']
        if split in ['val_fit','val_select']:
            index = ['val_fit','val_select'].index(split)
            assert sha(path) == calibration['validation_hashes'][model][index]
assert seen_ids == set(audit)
params = calibration['models']['jev']['selected_params']
assert params['T'] > 0 and all(math.isfinite(v) for v in params.values())
print('资料检查通过：6000 条训练数据和对应 Teacher 输出完整。')
print('Jev 校准参数：', params, '；类别顺序：E / N / C')


## 5．导入图片：自动下载 Flickr30K 原始压缩包

使用交接脚本中记录的**相同数据源、相同固定版本**。首次下载约 4.4 GB；文件保留在 Drive，下次可复用。仅在临时本机解压清单里的 7800 张，不重新编码。

下载失败或需要登录时，先处理网络/账号访问问题，再重跑本格；不要换其他版本凑数。如确实需要 Hugging Face 登录，在 Colab Secrets 添加 `HF_TOKEN`，不要把 token 写入代码。自动读取失败时会尝试匿名访问。


In [ ]:
from huggingface_hub import hf_hub_download
token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass
archive_dir = WORK / 'input_archive'
archive_dir.mkdir(exist_ok=True)
archive = Path(hf_hub_download(
    repo_id='nlphuji/flickr30k', repo_type='dataset',
    filename='flickr30k-images.zip',
    revision='2b239befc81b6e3f035ce6bd52f5f4d60f5625f7',
    local_dir=str(archive_dir), token=token))
assert zipfile.is_zipfile(archive), '下载内容不是有效 ZIP'
print('图片源压缩包已就绪：', archive)


## 6．筛选图片，并核对同学的原始图片哈希

按 `image_file` 找图，验证文件 SHA256 和解码后的 RGB 像素 SHA256。若不一致会立即停止，**不会覆盖原审计记录或悄悄换图**。本格可重跑，正确的已解压图片会复用。


In [ ]:
from PIL import Image
all_rows = [row for rows in manifest.values() for row in rows]
wanted = {r['image_file']:r for r in all_rows}
assert len(wanted) == 7800
with zipfile.ZipFile(archive) as z:
    members = {}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename.split('/'):
            continue
        name = Path(info.filename).name
        if name in wanted:
            assert name not in members, f'源包存在重复文件名：{name}'
            members[name] = info
    assert set(members) == set(wanted), '源包缺少清单图片'
    for index, (name, row) in enumerate(wanted.items(), 1):
        assert Path(name).name == name
        dest = IMAGES / name
        original = audit[row['sample_id']]
        if not dest.exists() or sha(dest) != original['file_sha256']:
            data = z.read(members[name])
            assert hashlib.sha256(data).hexdigest() == original['file_sha256'], f'原文件哈希不一致：{name}'
            temp = dest.with_suffix('.partial')
            temp.write_bytes(data); temp.replace(dest)
        with Image.open(dest) as im:
            rgb = im.convert('RGB')
            pixel_sha = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
        assert pixel_sha == original['pixel_sha256'], f'像素哈希不一致：{name}'
        if index % 500 == 0:
            print(f'已核对 {index}/7800 张图片', flush=True)
save_json(WORK/'images_verified.json', dict(count=7800,
    original_audit_sha256=sha(INPUT/'manifests/image_audit.jsonl'),
    source_revision='2b239befc81b6e3f035ce6bd52f5f4d60f5625f7'))
print('全部 7800 张图片与 Teacher 使用的原图一致。')


## 7．先做程序检查和断点恢复检查

这些是小规模合成数据测试，不产生正式实验结论。检查指标计算、文件写入和 Student 断点续训。若失败，应先处理报错，不继续正式训练。


In [ ]:
run('-m', 'unittest', '-v', 'test_study.py', 'test_runtime.py', 'test_student_resume.py')


## 8．提取 CLIP 特征（只做一次，三组共用）

现在只提取训练集和验证选择集。CLIP 对图片与假设分别编码，组合成 `[图像, 文本, 两者差的绝对值, 两者乘积]`，CLIP 权重不更新。

使用交接代码固定的 CLIP 版本，FP32 提取，每批 32 条。缓存写入 Drive；**已完成的划分可复用，某个划分中途断线则重跑该划分**。超过 77 token 的文本按原方案截断，并记录 ID。


In [ ]:
def ensure_features(split):
    path = FEATURES / f'{split}.npz'
    if not path.exists():
        print(f'开始提取 {split} 特征，请等待这个划分完成。', flush=True)
        run('student.py', 'features', '--images', IMAGES,
            '--manifests', INPUT/'manifests', '--out', FEATURES, '--splits', split)
    from student import load_features
    data = load_features(FEATURES, split, INPUT/'manifests')
    meta = json.loads((FEATURES/f'{split}_meta.json').read_text())
    assert meta['image_audit_sha256'] == sha(INPUT/'manifests/image_audit.jsonl')
    assert data['x'].dtype.name == 'float32'
    print(split, '特征形状：', data['x'].shape, '；截断文本：', len(meta['truncated_ids']))
    data.close()

for split in ['train', 'val_select']:
    ensure_features(split)


## 9．正式训练：三组 × 三个随机种子

|组别|学什么|
|---|---|
|CE|只用真实标签|
|KD_raw|真实标签 + 原始 Jev 的软目标|
|KD_calibrated|真实标签 + 校准 Jev 的软目标|

固定为 30 epochs、batch 128、AdamW、学习率 0.001、weight decay 0.0001。蒸馏 alpha=0.5、tau=2；tau 与 Teacher 校准温度 T 不同，代码只应用一次校准。

三个种子为 20261010 / 20261011 / 20261012。同一种子的三组使用相同初始化与训练预算。根据 `val_select` Macro-F1 选择各自 checkpoint，平局比较 NLL。

**每 5 个 epoch 保存恢复点。** 断线后按前面步骤恢复环境并重跑本格，会复用已完成组，并从上一个恢复点继续；最多重做尚未保存的部分。不要从不可信来源加载 `.resume.pt`，这里仅使用自己的运行产物。


In [ ]:
SEEDS = [20261010, 20261011, 20261012]
run('student.py', 'train', '--features', FEATURES,
    '--manifests', INPUT/'manifests', '--predictions', INPUT/'predictions',
    '--calibration', INPUT/'results/calibration.json', '--out', STUDENTS,
    '--epochs', 30, '--seeds', *SEEDS)
selection = json.loads((STUDENTS/'selection.json').read_text())
assert len(selection['configs']) == 9
print('九次训练完成，最佳模型已保存到：', STUDENTS)


## 10．查看验证结果和训练曲线

这里只看验证集。曲线用于检查训练是否正常，不代表测试性能。不要根据本轮结果不断更换种子。后面的正式测试使用全部九个预定 checkpoint，不挑表现最好的单个种子代表整组。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
selection = json.loads((STUDENTS/'selection.json').read_text())
table = pd.DataFrame(selection['configs'])[['method','seed','epoch','validation_macro_f1']]
display(table)
history = pd.DataFrame(read_rows(STUDENTS/'history.jsonl'))
history['macro_f1'] = history['validation'].map(lambda x:x['macro_f1'])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, seed in zip(axes, SEEDS):
    for method in ['CE','KD_raw','KD_calibrated']:
        part = history[(history.seed == seed) & (history.method == method)]
        ax.plot(part.epoch, part.macro_f1, label=method)
    ax.set_title(f'Seed {seed}'); ax.set_xlabel('Epoch'); ax.legend()
axes[0].set_ylabel('Validation Macro-F1')
fig.tight_layout()
fig.savefig(RESULTS/'validation_curves.png', dpi=160)
plt.show()


## 11．锁定实验，再运行最终测试

九次训练完成后，将下方 `RUN_FINAL_TEST = False` 改成 `True` 再运行。这个开关用于避免误点「全部运行」时过早查看测试结果。

本格锁定九个模型、特征、校准和训练配置，随后提取测试特征并评估。已有正式报告时只读检查，不重新覆盖。中途断线且报告尚未产生时，可用同一批锁定模型重跑。

测试集不用于训练、校准或 checkpoint 选择。它是固定的 1000 条子集，不是官方全量 benchmark。


In [ ]:
RUN_FINAL_TEST = False
if not RUN_FINAL_TEST:
    print('尚未启动最终测试。九次训练完成后，将本格开关改成 True。')
else:
    selection = json.loads((STUDENTS/'selection.json').read_text())
    expected_runs = {(m,s) for m in ['CE','KD_raw','KD_calibrated'] for s in SEEDS}
    assert len(selection['configs']) == 9
    assert {(c['method'],c['seed']) for c in selection['configs']} == expected_runs
    for c in selection['configs']:
        assert sha(STUDENTS/c['checkpoint']) == c['checkpoint_sha256']
    assert selection['calibration_sha256'] == sha(INPUT/'results/calibration.json')
    assert selection['teacher_train_sha256'] == sha(INPUT/'predictions/train/jev.jsonl')
    for split in ['train','val_select']:
        assert selection['feature_hashes'][split] == sha(FEATURES/f'{split}.npz')
    frozen = dict(selection_sha256=sha(STUDENTS/'selection.json'),
                  training_lock_sha256=sha(STUDENTS/'training_lock.json'),
                  test_manifest_sha256=sha(INPUT/'manifests/test.jsonl'),
                  clip_revision_sha256=sha(FEATURES/'clip_revision.json'),
                  protocol='3 methods x 3 seeds; all checkpoints; no test tuning')
    lock = RESULTS/'evaluation_lock.json'
    if lock.exists():
        assert json.loads(lock.read_text()) == frozen, '测试配置改变，停止。'
    else:
        save_json(lock, frozen)
    ensure_features('test')
    report_path = RESULTS/'student_test_report.json'
    if not report_path.exists():
        run('student.py', 'evaluate', '--features', FEATURES,
            '--manifests', INPUT/'manifests', '--students', STUDENTS, '--out', report_path)
    report = json.loads(report_path.read_text())
    assert report['selection_sha256'] == frozen['selection_sha256']
    print('正式测试报告已就绪：', report_path)


## 12．汇总最终结果：均值、标准差、三种配对差值

仅在第 11 节已完成后运行。Accuracy 和 Macro-F1 表格用百分数，差值用**百分点**。

重点比较：原始 KD − CE、校准 KD − CE、校准 KD − 原始 KD。均值 ± 标准差反映三个训练种子的波动；每个种子按图片配对 bootstrap 的区间反映测试样本不确定性，两者不是同一种统计量。

如果没有提升，也保留并报告结果。此 notebook 不预设哪组一定获胜。


In [ ]:
report_path = RESULTS/'student_test_report.json'
if not report_path.exists():
    print('还没有最终测试报告，请先完成第 11 节。')
else:
    import numpy as np
    from study import bootstrap_delta
    report = json.loads(report_path.read_text())
    records = []
    for method in ['CE','KD_raw','KD_calibrated']:
        for seed in SEEDS:
            score = report['scores'][f'{method}_{seed}']
            records.append(dict(method=method, seed=seed,
                accuracy=100*score['accuracy'], macro_f1=100*score['macro_f1']))
    per_seed = pd.DataFrame(records)
    summary = per_seed.groupby('method', sort=False)[['accuracy','macro_f1']].agg(['mean','std'])
    print('测试指标（%，std 为三个种子的样本标准差）：')
    display(summary.round(3))
    per_seed.to_csv(RESULTS/'test_per_seed.csv', index=False)
    summary.to_csv(RESULTS/'test_mean_std.csv')
    deltas = []
    for better, reference in [('KD_raw','CE'),('KD_calibrated','CE'),('KD_calibrated','KD_raw')]:
        for seed in SEEDS:
            a = report['scores'][f'{better}_{seed}']; b = report['scores'][f'{reference}_{seed}']
            deltas.append(dict(comparison=f'{better} minus {reference}', seed=seed,
                accuracy_pp=100*(a['accuracy']-b['accuracy']),
                macro_f1_pp=100*(a['macro_f1']-b['macro_f1'])))
    delta_table = pd.DataFrame(deltas)
    display(delta_table.round(3))
    delta_table.to_csv(RESULTS/'paired_deltas.csv', index=False)
    extra_path = RESULTS/'calibrated_vs_raw_bootstrap.json'
    if not extra_path.exists():
        extra = {}
        for seed in SEEDS:
            raw = read_rows(RESULTS/f'KD_raw_{seed}_test.jsonl')
            calibrated = read_rows(RESULTS/f'KD_calibrated_{seed}_test.jsonl')
            assert [r['sample_id'] for r in raw] == [r['sample_id'] for r in calibrated]
            assert [r['label'] for r in raw] == [r['label'] for r in calibrated]
            extra[str(seed)] = bootstrap_delta(
                np.array([r['label'] for r in raw]),
                np.array([r['probabilities'] for r in raw]),
                np.array([r['probabilities'] for r in calibrated]),
                np.array([r['image_id'] for r in raw]))
        save_json(extra_path, extra)
    print('报告和补充比较已保存：', RESULTS)


## 13．导出交付结果

生成一个结果 ZIP，包含模型、恢复记录、特征缓存、Student 报告、环境、清单、Teacher 输出及校准参数。不重复打包 4.4 GB 图片源压缩包。ZIP 保存在 Drive，可以右键下载。

交付时说明：冻结 CLIP + MLP、三组各三个种子、固定测试子集。模型推理时不需要 Teacher；真实速度和显存收益需要另行测量。


In [ ]:
if not (RESULTS/'student_test_report.json').exists():
    print('请先完成最终测试，再导出完整交付包。')
else:
    output_zip = WORK / 'student_delivery.zip'
    temp_zip = output_zip.with_suffix('.partial')
    with zipfile.ZipFile(temp_zip, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for folder in [INPUT, FEATURES, STUDENTS, RESULTS]:
            for file in sorted(folder.rglob('*')):
                if file.is_file() and '__pycache__' not in file.parts and not file.name.endswith('.partial'):
                    z.write(file, file.relative_to(WORK))
        for file in [WORK/'student_environment.json', WORK/'images_verified.json']:
            z.write(file, file.relative_to(WORK))
    temp_zip.replace(output_zip)
    print('交付包已保存到 Google Drive：', output_zip)
    print('文件大小：', round(output_zip.stat().st_size / 1024**2, 1), 'MB')


## 断线或报错时

- **安装后 import 异常：**先重启会话，再从第 2 节开始，确认安装没有报错。
- **下载 401/403：**检查该固定来源的访问权限和 Colab Secrets 中的 HF_TOKEN；不要把 token 发给别人。
- **图片哈希不一致：**停止，保留错误文件名和原审计记录，检查来源及解码环境；不要覆盖审计来绕过校验。
- **特征提取中断：**重连后重新运行第 2～8 节。完整划分复用，中断的划分重新计算。
- **训练中断：**恢复相同环境后重跑第 9 节，从最近每 5 epoch 的恢复点继续。
- **恢复提示环境或配置改变：**恢复原设置；若有意做新实验，使用新目录并明确记录实验变更。
- **最终测试中断：**保持九个 checkpoint 和测试锁定文件不变，重跑第 11 节；已有正式报告会复用。

不要把 smoke 或合成数据检查的输出当作真实模型效果。不要看完测试结果再修改本轮参数并覆盖报告。
